# Cargar Datos
**Proyecto:** Modelo de riesgo crediticio — PIM5 Data Science
**Objetivo de este notebook:** cargar la fuente de datos cruda, validar su integridad estructural (dimensiones, tipos, duplicados) y dejarla lista para el análisis exploratorio en `comprension_eda.ipynb`.

Este notebook **no transforma** los datos (eso ocurre en `ft_engineering.py`), solo carga y valida.

In [1]:
import json
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

## 1. Cargar configuración del proyecto

In [2]:
with open('config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

RAW_DATA_PATH = Path(config['raw_data_path'])
TARGET_COLUMN = config['target_column']

print(f"Proyecto: {config['project_code']}")
print(f"Ruta de datos: {RAW_DATA_PATH.resolve()}")
print(f"Variable objetivo: {TARGET_COLUMN}")

Proyecto: riesgo_crediticio
Ruta de datos: /home/claude/repo/Base_de_datos.csv
Variable objetivo: Pago_atiempo


## 2. Cargar el dataset crudo

In [3]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

Dimensiones: 10763 filas x 23 columnas


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,695.0,10,5,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,789.0,3,1,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,740.0,4,5,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,837.0,4,4,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,771.0,4,6,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


## 3. Validaciones estructurales básicas

In [4]:
print("Tipos de dato por columna:")
print(df.dtypes)

Tipos de dato por columna:
tipo_credito                       int64
fecha_prestamo                       str
capital_prestado                 float64
plazo_meses                        int64
edad_cliente                       int64
tipo_laboral                         str
salario_cliente                    int64
total_otros_prestamos              int64
cuota_pactada                      int64
puntaje                          float64
puntaje_datacredito              float64
cant_creditosvigentes              int64
huella_consulta                    int64
saldo_mora                       float64
saldo_total                      float64
saldo_principal                  float64
saldo_mora_codeudor              float64
creditos_sectorFinanciero          int64
creditos_sectorCooperativo         int64
creditos_sectorReal                int64
promedio_ingresos_datacredito    float64
tendencia_ingresos                   str
Pago_atiempo                       int64
dtype: object


In [5]:
n_dup = df.duplicated().sum()
print(f"Filas duplicadas: {n_dup}")

nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print("\nColumnas con valores nulos:")
print(nulls)

Filas duplicadas: 0

Columnas con valores nulos:
tendencia_ingresos               2932
promedio_ingresos_datacredito    2930
saldo_mora_codeudor               590
saldo_principal                   405
saldo_mora                        156
saldo_total                       156
puntaje_datacredito                 6
dtype: int64


In [6]:
assert TARGET_COLUMN in df.columns, f"No se encontró la columna objetivo '{TARGET_COLUMN}' en el dataset"
print(df[TARGET_COLUMN].value_counts(normalize=True) * 100)

Pago_atiempo
1    95.252253
0     4.747747
Name: proportion, dtype: float64


## 4. Guardar checkpoint

Guardamos una copia validada (misma info, solo confirmada) para que `comprension_eda.ipynb`
parta siempre del mismo punto de control.

In [7]:
OUTPUT_PATH = Path('../data_checkpoint.parquet')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)
print(f"Checkpoint guardado en: {OUTPUT_PATH.resolve()}")

Checkpoint guardado en: /home/claude/repo/mlops_pipeline/data_checkpoint.parquet
